---
# Commande pour le rendu : quarto render main.ipynb --execute
# Instructions pour quarto
title: "Traitement des données DVF Île-de-France"
format:
  html:
    code-fold: true
    embed-resources: true
---

# Introduction

- **Contexte :** nous exploitons les données des **Demandes de Valeurs Foncières (DVF)** pour analyser les prix des logements (appartements et maisons) en Île-de-France. Ces données fournissent des informations issues des actes notariés et des données cadastrales : valeur du bien, date de mutation, type de bien, surface, localisation, etc.
- **Objectif :** obtenir des **prix immobiliers précis au mètre carré**, par adresse et type de bien, afin de faciliter l’analyse du marché immobilier et son lien éventuel avec l’offre de transport.
- **Données :** nous utilisons les fichiers DVF couvrant les **cinq dernières années**. Ils sont disponibles en **format texte standardisé**, et la documentation complète est consultable [ici](https://eu.ftp.opendatasoft.com/stif/GTFS/opendata_gtfs.pdf).
    - Nettoyage des colonnes essentielles (code département, code postal, adresse)
    - Filtrage pour les départements d’Île-de-France
    - Reconstruction des adresses
    - Agrégation des transactions par adresse et date pour gérer les lots multiples
    - Calcul du **prix au mètre carré**
    - Filtrage des valeurs extrêmes pour éliminer les anomalies
    - Géolocalisation des logements via l’API Base Adresse Nationale (BAN)

# Chargement des données


In [44]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir('..')
import pandas as pd
import numpy as np
import requests

from script.download_data import get_IDFM_data_path

file_names = download_data.get_valeur_fonciere_path()

with open(file_names, "r", encoding="utf-8") as f:
    for i in range(10):
        print(f.readline().strip())

df = pd.read_csv(file_names, sep="|")

colonnes_utiles = [
    "Date mutation", 
    "Nature mutation", 
    "Valeur fonciere", 
    "No voie", 
    "Voie",
    "Type de voie",
    "Code voie",
    "Code postal", 
    "Commune",
    "B/T/Q",
    "Code commune",
    "Code departement",
    "Type local", 
    "Code type local",
    "Surface reelle bati", 
    "Nombre pieces principales", 
    "Surface terrain", 
    "1er lot",
    "2eme lot",
    "3eme lot",
    "4eme lot",
    "5eme lot"
]

df_filtre = df[colonnes_utiles]

Utilisation des données en cache dans cache/valeur_fonciere
Identifiant de document|Reference document|1 Articles CGI|2 Articles CGI|3 Articles CGI|4 Articles CGI|5 Articles CGI|No disposition|Date mutation|Nature mutation|Valeur fonciere|No voie|B/T/Q|Type de voie|Code voie|Voie|Code postal|Commune|Code departement|Code commune|Prefixe de section|Section|No plan|No Volume|1er lot|Surface Carrez du 1er lot|2eme lot|Surface Carrez du 2eme lot|3eme lot|Surface Carrez du 3eme lot|4eme lot|Surface Carrez du 4eme lot|5eme lot|Surface Carrez du 5eme lot|Nombre de lots|Code type local|Type local|Identifiant local|Surface reelle bati|Nombre pieces principales|Nature culture|Nature culture speciale|Surface terrain
|||||||000001|02/01/2024|Vente|346,50||||B020|LE DELIVRE|1230|CHALEY|01|76||B|514||||||||||||0||||||P||99
|||||||000002|03/01/2024|Vente|10000,00||||B007|CHEVRY DESSOUS|1170|CHEVRY|01|103||B|1782||||||||||||0||||||S||115
|||||||000001|08/01/2024|Vente|249000,00||||B086|PIN HAMEAU|1290

/tmp/ipykernel_137258/2990471556.py:16: DtypeWarning: Columns (18,23,24,26,28,30,31,33,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_names, sep="|")


# Traitement 

# Filtrage pour les départements d’Île-de-France

Les données DVF sont nationale et nous souhaitons concentrer nos analyse sur l'Île de France. Nous filtrons les département après avoir vérifié l'absence de valeurs manquantes dans la colonne du dataset et nettoyé celle-ci. 

In [45]:

#Nettoyage de la Colonne Code departement
df_filtre["Code departement"] = df_filtre["Code departement"].astype(str).str.strip()
# Filtrage des communes en ile de france 
codes_idf = ["75", "77", "78", "91", "92", "93", "94", "95"]
idf = df_filtre[df_filtre["Code departement"].isin(codes_idf)]

/tmp/ipykernel_137258/1379743959.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtre["Code departement"] = df_filtre["Code departement"].astype(str).str.strip()


# Reconstruction des adresses 

Nous reconstruson l'adresse des logement à partir des information mis à notre disposition : Numéro de voie, voie et type de voie

In [46]:
idf["x"] = pd.NA
idf["y"] = pd.NA

idf["Code postal"] = idf["Code postal"].astype(str).str.replace(".0", "", regex=False).str.strip()
idf["No voie"] = idf["No voie"].astype(str).str.replace(".0", "", regex=False).str.strip()

idf["Type de voie"] = idf["Type de voie"].fillna("").astype(str).str.strip()
idf["Voie"] = idf["Voie"].fillna("").astype(str).str.strip()

# Créer une colonne adresse propre
idf["adresse"] = (
    idf["No voie"] + " " +
    idf["Type de voie"] + " " +
    idf["Voie"]
).str.replace(" +", " ", regex=True).str.strip()

/tmp/ipykernel_137258/706913884.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  idf["x"] = pd.NA
/tmp/ipykernel_137258/706913884.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  idf["y"] = pd.NA
/tmp/ipykernel_137258/706913884.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

# Agrégation des transactions par adresse et date pour gérer les lots multiples

Certaines transactions immobilières dans les fichiers DVF concernent plusieurs lots (par exemple, un immeuble vendu en plusieurs parties ou un bien cadastral divisé).  
Si chaque lot était conservé séparément, cela entraînerait un **double comptage** des biens et biaiserait les calculs de surface totale, de nombre de pièces et du prix au mètre carré.  

Pour corriger cela, nous effectuons une **agrégation par adresse et date de mutation**.  
Chaque groupe de lignes correspondant à une même adresse et à la même date est considéré comme une **transaction unique**.  

Lors de cette agrégation :  
- La **valeur foncière** est soit conservée telle quelle si identique pour tous les lots, soit sommée pour refléter l’ensemble des lots.  
- Le **code postal** et la **commune** sont pris à partir de la première valeur du groupe, car elles sont identiques pour tous les lots.  
- La **surface réelle bâtie**, la **surface du terrain** et le **nombre de pièces principales** sont **sommées** pour obtenir les totaux correspondant à l’ensemble des lots.  
- Le **code commune** et le **type de local** sont pris à partir de la première valeur non manquante, afin de préserver l’information principale du bien.  

Cette opération garantit que chaque transaction est comptabilisée **une seule fois**, et que les mesures de surface et de composition du bien sont exactes.  
C’est une étape essentielle pour calculer un **prix au mètre carré fiable** et pour réaliser des analyses statistiques précises.

In [47]:
def valeur_fonciere_agg(x):
    if len(x.unique()) == 1:
        return x.iloc[0]
    else:
        return pd.to_numeric(x, errors='coerce').sum()

def sum_numeric(x):
    return pd.to_numeric(x, errors='coerce').sum()

idf = (
    idf
    .groupby(["adresse", "Date mutation"], as_index=False)
    .agg({
        "Valeur fonciere": valeur_fonciere_agg,
        "Code postal": "first",
        "Commune": "first",
        "Surface reelle bati": sum_numeric,
        "Surface terrain": sum_numeric,
        "Nombre pieces principales": sum_numeric,
        "Code commune": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA,
        "Type local": lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else pd.NA
    })
)

# Nettoyage des données et traitement des valeurs aberrantes

Cette étape vise à nettoyer les données DVF et à éliminer les **valeurs aberrantes** afin d’obtenir un indicateur de **valeur foncière au mètre carré** fiable et exploitable pour l’analyse.

## Filtrage des biens et des surfaces
- Suppression des observations dont la **surface réelle bâtie** est manquante.
- Conversion de la surface réelle bâtie en format numérique.
- Conservation uniquement des biens dont la surface est comprise entre **9 m² et 300 m²**, afin d’exclure les valeurs irréalistes.
- Sélection des **maisons et appartements** uniquement, pour se concentrer sur le résidentiel.
- Suppression des lignes avec une **valeur foncière manquante**.

## Nettoyage de la valeur foncière
La variable **valeur foncière** est nettoyée pour garantir un format numérique homogène 

## Calcul de la valeur foncière au mètre carré
La **valeur foncière au mètre carré** est calculée comme le rapport entre la valeur foncière et la surface réelle bâtie.

## Analyse des distributions avant filtrage
- Calcul des **déciles** de la valeur foncière au mètre carré afin d’observer la distribution.
- Identification des valeurs minimale et maximale.
- Comptage du nombre d’observations avant filtrage.

## Traitement des valeurs aberrantes
Afin d’éliminer les prix au mètre carré incohérents ou extrêmes :
- Conservation uniquement des biens dont la valeur foncière au mètre carré est comprise entre **1 000 € et 20 000 €**.
- Suppression des observations situées en dehors de cet intervalle.

## Analyse après filtrage
- Recalcul des **déciles** après suppression des valeurs aberrantes.
- Comparaison du nombre d’observations avant et après filtrage.
- Vérification des nouvelles valeurs minimale et maximale.

Cette démarche permet de réduire l’influence des valeurs extrêmes, d’améliorer la robustesse des statistiques descriptives et d’assurer la fiabilité des analyses de prix immobiliers.


In [48]:
idf = idf[idf["Surface reelle bati"].notna()]
idf["Surface reelle bati"] = pd.to_numeric(idf["Surface reelle bati"])
idf = idf[(idf["Surface reelle bati"] >= 9) & (idf["Surface reelle bati"] <= 300)]

idf = idf[idf["Type local"].isin(["Maison", "Appartement"])]

idf = idf.dropna(subset=["Valeur fonciere"])

idf["Valeur fonciere"] = (
    idf["Valeur fonciere"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.replace("\xa0", "", regex=False)
        .str.extract(r'(\d+\.?\d*)')[0]
        .astype(float)
)

idf["Valeur foncière au mètre carré"] = (
    idf["Valeur fonciere"] / idf["Surface reelle bati"]
)

deciles_avant = idf["Valeur foncière au mètre carré"].quantile([0.1 * i for i in range(1, 11)])
deciles_avant = deciles_avant.apply(lambda x: f"{x:,.2f} €")

n_before = len(idf)
prix_min_avant = idf["Valeur foncière au mètre carré"].min()
prix_max_avant = idf["Valeur foncière au mètre carré"].max()

idf = idf.loc[
    (idf["Valeur foncière au mètre carré"] >= 1000) &
    (idf["Valeur foncière au mètre carré"] <= 20000)
]

deciles_apres = idf["Valeur foncière au mètre carré"].quantile([0.1 * i for i in range(1, 11)])
deciles_apres = deciles_apres.apply(lambda x: f"{x:,.2f} €")

n_after = len(idf)
n_supprimees = n_before - n_after
prix_min_apres = idf["Valeur foncière au mètre carré"].min()
prix_max_apres = idf["Valeur foncière au mètre carré"].max()

In [49]:
print(f"Nombre de lignes avant filtrage : {n_before}")
print(f"Valeur foncière au mètre carré minimale avant filtrage : {prix_min_avant}")
print(f"Valeur foncière au mètre carré maximal avant filtrage : {prix_max_avant}")

Nombre de lignes avant filtrage : 75944
Valeur foncière au mètre carré minimale avant filtrage : 0.0
Valeur foncière au mètre carré maximal avant filtrage : 2247428.5714285714


In [50]:
print("Déciles avant filtrage :")
print(deciles_avant.to_string())

Déciles avant filtrage :
0.1        2,000.00 €
0.2        2,727.27 €
0.3        3,238.81 €
0.4        3,750.00 €
0.5        4,384.62 €
0.6        5,298.41 €
0.7        6,666.67 €
0.8        8,317.48 €
0.9       10,333.33 €
1.0    2,247,428.57 €


In [51]:
print(f"Nombre de lignes supprimées : {n_supprimees}")
print(f"Valeur foncière au mètre carré minimale après filtrage : {prix_min_apres}")
print(f"Valeur foncière au mètre carré maximal avant filtrage : {prix_max_apres}")

Nombre de lignes supprimées : 3010
Valeur foncière au mètre carré minimale après filtrage : 1000.0
Valeur foncière au mètre carré maximal avant filtrage : 20000.0


In [52]:
print("\nDéciles après filtrage :")
print(deciles_apres.to_string())


Déciles après filtrage :
0.1     2,261.75 €
0.2     2,860.47 €
0.3     3,341.00 €
0.4     3,846.17 €
0.5     4,485.71 €
0.6     5,400.00 €
0.7     6,750.00 €
0.8     8,333.33 €
0.9    10,277.78 €
1.0    20,000.00 €


On affiche les percentiles : 

In [53]:
percentiles = np.arange(1, 100, 1)  # 1%, 2%, ..., 99%
valeurs_percentiles = idf["Valeur foncière au mètre carré"].quantile(percentiles / 100.0)
for p, v in zip(percentiles, valeurs_percentiles):
    print(f"Percentile {p}% : {v:,.2f} €/m²")

Percentile 1% : 1,234.15 €/m²
Percentile 2% : 1,400.00 €/m²
Percentile 3% : 1,555.56 €/m²
Percentile 4% : 1,685.39 €/m²
Percentile 5% : 1,803.28 €/m²
Percentile 6% : 1,904.76 €/m²
Percentile 7% : 2,000.00 €/m²
Percentile 8% : 2,093.75 €/m²
Percentile 9% : 2,178.85 €/m²
Percentile 10% : 2,261.75 €/m²
Percentile 11% : 2,333.33 €/m²
Percentile 12% : 2,400.00 €/m²
Percentile 13% : 2,465.75 €/m²
Percentile 14% : 2,530.00 €/m²
Percentile 15% : 2,589.29 €/m²
Percentile 16% : 2,647.06 €/m²
Percentile 17% : 2,703.70 €/m²
Percentile 18% : 2,758.62 €/m²
Percentile 19% : 2,812.50 €/m²
Percentile 20% : 2,860.47 €/m²
Percentile 21% : 2,910.11 €/m²
Percentile 22% : 2,961.23 €/m²
Percentile 23% : 3,006.97 €/m²
Percentile 24% : 3,055.56 €/m²
Percentile 25% : 3,105.92 €/m²
Percentile 26% : 3,152.54 €/m²
Percentile 27% : 3,202.50 €/m²
Percentile 28% : 3,250.00 €/m²
Percentile 29% : 3,295.45 €/m²
Percentile 30% : 3,341.00 €/m²
Percentile 31% : 3,391.30 €/m²
Percentile 32% : 3,437.50 €/m²
Percentile 33% : 

# Géocodage des adresses avec l'API Adresse de data.gouv.fr

# Géocodage des adresses avec l'API Adresse

Cette étape permet de **transformer les adresses textuelles en coordonnées géographiques** (latitude et longitude) via l’API officielle française `https://api-adresse.data.gouv.fr`.

## Étapes principales

1. **Nettoyage et uniformisation des adresses**  
   - Suppression des arrondissements de Paris (`PARIS 1`, `PARIS 2`, … → `PARIS`).  
   - Renommage et vérification des colonnes : `Code postal`, `adresse`, `Commune`.

2. **Création d’une colonne "search"**  
   - Combinaison de l’adresse, du code postal et de la commune pour former une chaîne unique.  
   - Cette colonne servira de référence pour le géocodage.

3. **Exportation vers CSV**  
   - Les adresses préparées sont sauvegardées dans un fichier `idfs.csv`.  
   - Le CSV sera envoyé à l’API pour traitement en lot.

4. **Envoi du fichier à l’API**  
   - Une requête POST transmet le CSV à l’API `https://api-adresse.data.gouv.fr/search/csv/`.  
   - L’API retourne un CSV géocodé avec les coordonnées et les informations normalisées.

5. **Gestion de la réponse**  
   - Si la requête réussit (`status_code == 200`), le CSV géocodé est sauvegardé localement (`idfs_geocoded.csv`).  
   - Sinon, le script affiche un message d’erreur.

6. **Lecture du CSV géocodé**  
   - Le fichier retourné est lu dans un DataFrame `idfs_geocoded` pour les analyses spatiales ou cartographiques.




In [54]:
idf["Commune"] = idf["Commune"].str.replace(r"PARIS \d{1,2}", "PARIS", regex=True)

idf = idf.rename(columns={
    "Code postal": "Code_postal",
    "adresse": "adresse",   # juste pour s'assurer que c'est correct
    "Commune": "Commune"
})

idf["search"] = idf["adresse"].astype(str) + " " + idf["Code_postal"].astype(str) + " " + idf["Commune"]
idf_search = idf["search"]
idf_search.to_csv("idfs.csv", index=False, encoding="utf-8")


payload = {
    "indexes": ["address"],
}

url = "https://api-adresse.data.gouv.fr/search/csv/"
files = {"data": open("idfs.csv", "rb")}
response = requests.post(url, files=files)

if response.status_code == 200:
    # Sauvegarde le CSV retourné tel quel
    with open("idfs_geocoded.csv", "wb") as f:
        f.write(response.content)
    print("CSV géocodé sauvegardé avec succès !")
else:
    print("Erreur API :", response.text)

idfs_geocoded = pd.read_csv("idfs_geocoded.csv")

idf = idf.merge(
    idfs_geocoded,
    on=["search"], 
    how="left"
)

na_counts = idf[["latitude", "longitude"]].isna().sum()
print(na_counts)

# Retirer les lignes avec NaN dans latitude ou longitude
idf = idf.dropna(subset=["latitude", "longitude"])



CSV géocodé sauvegardé avec succès !


/tmp/ipykernel_137258/4019363905.py:30: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  idfs_geocoded = pd.read_csv("idfs_geocoded.csv")


latitude     112
longitude    112
dtype: int64


# Visualisation

Une première visualisation nous permet de vérifier la distribution et la cohérence de nos données.

On utilise `folium` pour cartographier les maison et appartement dont nous disposons du prix et colorer les points en fonction de leur valeur foncière par metres carrés.

1. Creation du fond de carte :

In [55]:
# Creation d'un fond de carte avec les limites des departements et communes d'IDF
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
from script.download_fond_carte import load_fonds_carte

coms, deps = load_fonds_carte(crs=4326, force_download=False)

# Listes filtrées idf
PETITE_COURONNE = [75, 77, 78, 91, 92, 93, 94, 95]
coms_pc = coms[coms["INSEE_DEP"].astype(int).isin(PETITE_COURONNE)].copy()
deps_pc = deps[deps["INSEE_DEP"].astype(int).isin(PETITE_COURONNE)].copy()


# trace ces departements et communes avec folium
m = folium.Map(location=[48.84, 2.35], zoom_start=11, tiles="cartodb positron")

for _, row in deps_pc.iterrows():
    folium.GeoJson(
        row['geometry'],
        name=f"Departement {row['INSEE_DEP']}",
        style_function=lambda x: {
            'fillColor': 'none',
            'color': 'black',
            'weight': 2,
            'dashArray': '5, 5'
        }
    ).add_to(m)
for _, row in coms_pc.iterrows():
    folium.GeoJson(
        row['geometry'],
        name=f"Commune {row['INSEE_COM']}",
        style_function=lambda x: {
            'fillColor': 'none',
            'color': 'grey',
            'weight': 1,
            'dashArray': '2, 2'
        }
    ).add_to(m)

Téléchargement des bordures de communes / départements mis en cache


In [ ]:
import folium
import branca.colormap as cm
from numpy import log

idf_sample = idf.sample(frac = 0.1, random_state= 42)

cmap = cm.LinearColormap(
    colors=['blue', 'red'],
    vmin=idf_sample['Valeur foncière au mètre carré'].min(),
    vmax=idf_sample['Valeur foncière au mètre carré'].max(),
    caption='Valeur foncière au m² (€)'
)

for _, row in idf_sample.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],  # Remplace par tes colonnes lat/lon
        radius=4,
        tooltip=f"{row['adresse']} : {row['Valeur foncière au mètre carré']:.0f} €/m²",
        color=cmap(row['Valeur foncière au mètre carré']),
        fill=True,
        fill_color=cmap(row['Valeur foncière au mètre carré']),
        fill_opacity=0.7
    ).add_to(m)

# Ajouter la légende
cmap.add_to(m)

# Afficher la carte dans Jupyter Notebook
m
